# Phase 3 v2.0: Apply Feature Removal

## Objective

Remove 7 redundant features identified in Phase 3 notebook 01, based on:
1. **Perfect correlations** (r = 1.0 or -1.0) - Remove redundant, keep stronger predictor
2. **High correlations** (r > 0.9) - Remove redundant, keep more useful feature
3. **Raw metadata strings** - Already encoded in derived features

**Input**: Phase 2B Enhanced output (283,118 records, 62 columns)  
**Output**: Optimized feature set (283,118 records, 55 columns)

---

## Features to Remove (7)

### Perfect/High Correlations (5)
1. `creation_time_delta_days` - Perfectly correlated with `accessed_time_delta_days` (r = 1.0), but accessed is stronger predictor (r = 0.654 vs 0.138 with target)
2. `modified_time_delta_days` - Perfectly correlated with `event_vs_modified_after_days` (r = -1.0), keep event_vs
3. `source_confidence_score` - Highly correlated with `cross_artifact_validation_score` (r = 0.984), keep cross_artifact (r = 0.017 vs 0.013 with target)
4. `time_until_next_event_seconds` - Highly correlated with `time_since_previous_event_seconds` (r = 0.955), keep time_since
5. `events_in_1min_window` - Highly correlated with `events_in_5min_window` (r = 0.928), keep 5min window

### Raw Metadata Strings (1)
6. `usn_file_attribute` - Raw attribute string, already encoded in binary features (is_executable, is_system_file, is_hidden_file, is_archive)
---

## Features PRESERVED (based on forensic requirements)

### Forensic Identifiers (CRITICAL)
- `lf_lsn` - LogFile sequence number, forensic identifier for $LogFile events
- `lf_event` - Event type (SetInfo, CreateFile, etc.), provides context on what operation occurred
- `lf_target_vcn`, `lf_cluster_index` - Required by Oh et al. (2024) Algorithm 2

### Parsed Timestamps (temporal context)
- All 8 `lf_*_time_before/after` columns - Show actual timestamp values, not just magnitude

### Delta Columns (magnitude)
- `accessed_time_delta_days` - **Strongest predictor** (r = 0.654 with target)
- `mft_modified_time_delta_days` - Shows MFT metadata manipulation magnitude
- `event_vs_modified_after_days` - Shows temporal relationship

### Low-Variance Forensic Indicators (detect rare events)
- `has_attribute_change` - APT technique (variance = 0.0018)
- `has_timestamp_copied_from_file` - APT technique (variance = 0.0029)
- `zero_nanoseconds_logfile` - Manipulation indicator (variance = 0.0053)
- `file_system_tunneling_confidence` - Helps filter false positives (variance = 0.0089)

---

## 1. Setup & Load Data

In [37]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully")

Libraries imported successfully


In [38]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 2B - V2 Column Cleanup'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 3 - V2 Feature Selection'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Output exists: {OUTPUT_DIR.exists()}")

Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2B - V2 Column Cleanup
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection
  Output exists: True


In [39]:
# Load Phase 2B Enhanced output
print("Loading Phase 2B Enhanced dataset...")
input_file = INPUT_DIR / 'all_cases_combined_v2_phase2b_enhanced.csv'

df = pd.read_csv(input_file, encoding='utf-8-sig')

print(f"\nDataset loaded successfully:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Timestomped events: {(df['timestomped'] == 1).sum():,}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Loading Phase 2B Enhanced dataset...

Dataset loaded successfully:
  Records: 283,118
  Columns: 62
  Timestomped events: 280
  Memory usage: 437.55 MB


In [40]:
print("=" * 80)
print("STANDARDIZING CASE IDs")
print("=" * 80)

print(f"\nBefore standardization:")
print(f"  Unique case_ids: {df['case_id'].nunique()}")
print(f"  Case_id types: {df['case_id'].apply(type).unique()}")

# Show current case IDs
print(f"\nCurrent case IDs:")
for cid in sorted(df['case_id'].unique(), key=str):
    count = (df['case_id'] == cid).sum()
    print(f"  {repr(cid):30s} - {count:,} records")

# Standardize: Convert all to string and strip whitespace
df['case_id'] = df['case_id'].astype(str).str.strip()

print(f"\nAfter standardization:")
print(f"  Unique case_ids: {df['case_id'].nunique()}")
print(f"  Case_id types: {df['case_id'].apply(type).unique()}")

print(f"\nStandardized case IDs:")
for cid in sorted(df['case_id'].unique()):
    count = (df['case_id'] == cid).sum()
    print(f"  {cid:30s} - {count:,} records")

print("\n✓ Case IDs standardized to strings")


STANDARDIZING CASE IDs

Before standardization:
  Unique case_ids: 20
  Case_id types: [<class 'int'> <class 'str'>]

Current case IDs:
  '01-APT17'                     - 23,135 records
  '02-APT19'                     - 23,526 records
  '04-APT28'                     - 23,200 records
  '05-APT29'                     - 23,843 records
  1                              - 24,204 records
  10                             - 17,678 records
  '10-DarkHotel663'              - 17,446 records
  11                             - 3,553 records
  '11'                           - 1,736 records
  '11-DarkHotelbbd'              - 17,418 records
  '12'                           - 768 records
  12                             - 4,590 records
  2                              - 16,968 records
  3                              - 16,889 records
  4                              - 4,952 records
  5                              - 5,311 records
  6                              - 5,307 records
  7                    

---
## 2. Define Features to Remove

In [41]:
print("=" * 80)
print("FEATURES TO REMOVE")
print("=" * 80)

# Define the 7 features to remove
features_to_remove = [
    # Perfect/High Correlations (5)
    'creation_time_delta_days',           # r = 1.0 with accessed_time_delta_days
    'modified_time_delta_days',           # r = -1.0 with event_vs_modified_after_days
    'source_confidence_score',            # r = 0.984 with cross_artifact_validation_score
    'time_until_next_event_seconds',      # r = 0.955 with time_since_previous_event_seconds
    'events_in_1min_window',              # r = 0.928 with events_in_5min_window
    
    # Raw Metadata Strings (1)
    'usn_file_attribute',                 # Already encoded in binary features
]

print(f"\nTotal features to remove: {len(features_to_remove)}")
print("\nFeatures:")
for i, feat in enumerate(features_to_remove, 1):
    if feat in df.columns:
        coverage = (df[feat].notna().sum() / len(df)) * 100
        print(f"  {i}. {feat:45s} - {coverage:5.1f}% coverage")
    else:
        print(f"  {i}. {feat:45s} - NOT FOUND IN DATASET")

FEATURES TO REMOVE

Total features to remove: 6

Features:
  1. creation_time_delta_days                      -   0.2% coverage
  2. modified_time_delta_days                      -   1.0% coverage
  3. source_confidence_score                       - 100.0% coverage
  4. time_until_next_event_seconds                 - 100.0% coverage
  5. events_in_1min_window                         - 100.0% coverage
  6. usn_file_attribute                            -  99.8% coverage


---
## 3. Verify Features Exist

In [42]:
print("=" * 80)
print("VERIFICATION")
print("=" * 80)

# Check which features exist in the dataset
missing_features = [feat for feat in features_to_remove if feat not in df.columns]
existing_features = [feat for feat in features_to_remove if feat in df.columns]

print(f"\nFeatures to remove: {len(features_to_remove)}")
print(f"  - Found in dataset: {len(existing_features)}")
print(f"  - Not found: {len(missing_features)}")

if missing_features:
    print("\nWARNING: The following features are not in the dataset:")
    for feat in missing_features:
        print(f"  - {feat}")
else:
    print("\n✓ All features found in dataset")

VERIFICATION

Features to remove: 6
  - Found in dataset: 6
  - Not found: 0

✓ All features found in dataset


---
## 4. Remove Features

In [43]:
print("=" * 80)
print("REMOVING FEATURES")
print("=" * 80)

print(f"\nBefore removal:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# Remove features that exist
df_cleaned = df.drop(columns=existing_features)

print(f"\nAfter removal:")
print(f"  Records: {len(df_cleaned):,}")
print(f"  Columns: {len(df_cleaned.columns)}")
print(f"  Memory: {df_cleaned.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\nColumns removed: {len(df.columns) - len(df_cleaned.columns)}")
print(f"Memory saved: {(df.memory_usage(deep=True).sum() - df_cleaned.memory_usage(deep=True).sum()) / (1024**2):.2f} MB")

REMOVING FEATURES

Before removal:
  Records: 283,118
  Columns: 62
  Memory: 439.61 MB

After removal:
  Records: 283,118
  Columns: 56
  Memory: 412.42 MB

Columns removed: 6
Memory saved: 27.19 MB


---
## 5. Verify Preserved Features

In [44]:
print("=" * 80)
print("VERIFYING PRESERVED FEATURES")
print("=" * 80)

# Features that MUST be preserved
critical_features = [
    # Forensic identifiers
    'lf_lsn',
    'lf_event',
    'lf_target_vcn',
    'lf_cluster_index',
    
    # Parsed timestamps
    'lf_creation_time_before',
    'lf_creation_time_after',
    'lf_modified_time_before',
    'lf_modified_time_after',
    'lf_accessed_time_before',
    'lf_accessed_time_after',
    'lf_mft_modified_time_before',
    'lf_mft_modified_time_after',
    
    # Delta columns (strongest predictors)
    'accessed_time_delta_days',
    'mft_modified_time_delta_days',
    'event_vs_modified_after_days',
    
    # Low-variance forensic indicators
    'has_attribute_change',
    'has_timestamp_copied_from_file',
    'zero_nanoseconds_logfile',
    'file_system_tunneling_confidence',
    
    # Target
    'timestomped'
]

print(f"\nVerifying {len(critical_features)} critical features are preserved...")

preserved_count = 0
missing_count = 0

for feat in critical_features:
    if feat in df_cleaned.columns:
        preserved_count += 1
    else:
        missing_count += 1
        print(f"  ✗ MISSING: {feat}")

if missing_count == 0:
    print(f"\n✓ All {preserved_count} critical features preserved")
else:
    print(f"\n⚠ {missing_count} critical features are missing!")

VERIFYING PRESERVED FEATURES

Verifying 20 critical features are preserved...

✓ All 20 critical features preserved


---
## 6. Column Categorization

In [45]:
print("=" * 80)
print("FINAL COLUMN CATEGORIZATION")
print("=" * 80)

# Identifier columns
identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']

# Target variable
target_col = 'timestomped'

# Feature columns
feature_cols = [col for col in df_cleaned.columns if col not in identifier_cols + [target_col]]

# Numeric features
numeric_features = df_cleaned[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Non-numeric features
non_numeric_features = [col for col in feature_cols if col not in numeric_features]

print(f"\nColumn breakdown:")
print(f"  Identifiers: {len(identifier_cols)}")
print(f"  Target: 1")
print(f"  Features: {len(feature_cols)}")
print(f"    - Numeric: {len(numeric_features)}")
print(f"    - Non-numeric: {len(non_numeric_features)}")
print(f"\nTotal columns: {len(df_cleaned.columns)}")

FINAL COLUMN CATEGORIZATION

Column breakdown:
  Identifiers: 6
  Target: 1
  Features: 49
    - Numeric: 18
    - Non-numeric: 31

Total columns: 56


---
## 7. Data Quality Check

In [46]:
print("=" * 80)
print("DATA QUALITY CHECK")
print("=" * 80)

# Check target variable distribution
print(f"\nTarget variable distribution:")
print(f"  Timestomped (1): {(df_cleaned['timestomped'] == 1).sum():,} ({(df_cleaned['timestomped'] == 1).sum() / len(df_cleaned) * 100:.2f}%)")
print(f"  Normal (0): {(df_cleaned['timestomped'] == 0).sum():,} ({(df_cleaned['timestomped'] == 0).sum() / len(df_cleaned) * 100:.2f}%)")

# Check for duplicates
duplicates = df_cleaned.duplicated().sum()
print(f"\nDuplicate rows: {duplicates:,}")

# Check for null values in target
null_target = df_cleaned['timestomped'].isnull().sum()
print(f"Null values in target: {null_target:,}")

# Memory usage
print(f"\nMemory usage: {df_cleaned.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

DATA QUALITY CHECK

Target variable distribution:
  Timestomped (1): 280 (0.10%)
  Normal (0): 282,838 (99.90%)



Duplicate rows: 0
Null values in target: 0

Memory usage: 412.42 MB


---
## 8. Save Cleaned Dataset

In [47]:
print("=" * 80)
print("SAVING CLEANED DATASET")
print("=" * 80)

# Save to CSV
output_file = OUTPUT_DIR / 'all_cases_combined_v2_phase3_final.csv'

print(f"\nSaving to: {output_file}")
df_cleaned.to_csv(output_file, index=False, encoding='utf-8-sig')

# Verify file was saved
if output_file.exists():
    file_size = output_file.stat().st_size / (1024**2)
    print(f"\n✓ File saved successfully")
    print(f"  Size: {file_size:.2f} MB")
    print(f"  Location: {output_file}")
else:
    print(f"\n✗ ERROR: File was not saved")

SAVING CLEANED DATASET

Saving to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection/all_cases_combined_v2_phase3_final.csv

✓ File saved successfully
  Size: 148.05 MB
  Location: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection/all_cases_combined_v2_phase3_final.csv


---
## 9. Summary Report

In [48]:
print("\n" + "=" * 80)
print("PHASE 3 v2.0 SUMMARY REPORT - FEATURE REMOVAL")
print("=" * 80)

print("\nPHASE 3 v2.0 COMPLETE - FEATURE REMOVAL APPLIED")

print("\n" + "=" * 80)
print("1. DATASET CHANGES")
print("=" * 80)
print(f"  Input:")
print(f"    Records: {len(df):,}")
print(f"    Columns: {len(df.columns)}")
print(f"  Output:")
print(f"    Records: {len(df_cleaned):,}")
print(f"    Columns: {len(df_cleaned.columns)}")
print(f"  Changes:")
print(f"    Columns removed: {len(df.columns) - len(df_cleaned.columns)}")

print("\n" + "=" * 80)
print("2. FEATURES REMOVED (6)")
print("=" * 80)
print(f"  Perfect/High Correlations: 5")
for feat in existing_features[:5]:
    print(f"    - {feat}")
print(f"  Raw Metadata Strings: 1")
for feat in existing_features[5:]:
    print(f"    - {feat}")

print("\n" + "=" * 80)
print("3. CRITICAL FEATURES PRESERVED")
print("=" * 80)
print(f"  Forensic identifiers: lf_lsn, lf_event, lf_target_vcn, lf_cluster_index")
print(f"  Parsed timestamps: All 8 lf_*_time_before/after columns")
print(f"  Delta columns: accessed_time_delta_days (r=0.654), mft_modified_time_delta_days, event_vs_modified_after_days")
print(f"  Forensic indicators: has_attribute_change, has_timestamp_copied_from_file, zero_nanoseconds_logfile, file_system_tunneling_confidence")

print("\n" + "=" * 80)
print("4. FINAL FEATURE SET")
print("=" * 80)
print(f"  Total columns: {len(df_cleaned.columns)}")
print(f"  Identifiers: {len(identifier_cols)}")
print(f"  Target: 1")
print(f"  Features: {len(feature_cols)}")
print(f"    - Numeric: {len(numeric_features)}")
print(f"    - Non-numeric: {len(non_numeric_features)}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("\n1. Proceed to Phase 4: Model Training")
print("2. Train models on optimized feature set")
print("3. Evaluate model performance")
print("4. Compare with Phase 1 baseline")

print("\n" + "=" * 80)
print("OUTPUT FILE")
print("=" * 80)
print(f"  {output_file}")
print("=" * 80)


PHASE 3 v2.0 SUMMARY REPORT - FEATURE REMOVAL

PHASE 3 v2.0 COMPLETE - FEATURE REMOVAL APPLIED

1. DATASET CHANGES
  Input:
    Records: 283,118
    Columns: 62
  Output:
    Records: 283,118
    Columns: 56
  Changes:
    Columns removed: 6

2. FEATURES REMOVED (6)
  Perfect/High Correlations: 5
    - creation_time_delta_days
    - modified_time_delta_days
    - source_confidence_score
    - time_until_next_event_seconds
    - events_in_1min_window
  Raw Metadata Strings: 1
    - usn_file_attribute

3. CRITICAL FEATURES PRESERVED
  Forensic identifiers: lf_lsn, lf_event, lf_target_vcn, lf_cluster_index
  Parsed timestamps: All 8 lf_*_time_before/after columns
  Delta columns: accessed_time_delta_days (r=0.654), mft_modified_time_delta_days, event_vs_modified_after_days
  Forensic indicators: has_attribute_change, has_timestamp_copied_from_file, zero_nanoseconds_logfile, file_system_tunneling_confidence

4. FINAL FEATURE SET
  Total columns: 56
  Identifiers: 6
  Target: 1
  Features: